<a href="https://colab.research.google.com/github/Ayush-Gole8/DeepLearning/blob/main/23102A0005_DL_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate rouge-score accelerate sentencepiece

"""
IMPORT REQUIRED LIBRARIES
"""
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

"""
DATASET PREPARATION AND SCALING
"""
dataset = load_dataset("ccdv/pubmed-summarization", "section", split="train[:1500]")
dataset = dataset.train_test_split(test_size=0.1)

"""
TOKENIZATION PIPELINE
"""
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples["abstract"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

"""
MODEL ARCHITECTURE INITIALIZATION
"""
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

"""
EVALUATION METRICS CONFIGURATION
"""
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    vocab_size = tokenizer.vocab_size
    predictions = np.clip(predictions, 0, vocab_size - 1).astype(np.int64)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, vocab_size - 1).astype(np.int64)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

"""
ADVANCED TRAINING CONFIGURATION (COLAB OPTIMIZED)
"""
training_args = Seq2SeqTrainingArguments(
    output_dir="./medical_summary_model_optimized",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

"""
EXECUTE RIGOROUS TRAINING
"""
trainer.train()

"""
QUALITATIVE INFERENCE AND COMPARISON
"""
sample_index = 0
sample_report = dataset["test"][sample_index]["article"]
human_summary = dataset["test"][sample_index]["abstract"]

input_text = "summarize: " + sample_report
input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids.to(model.device)

outputs = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
generated_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*80)
print("ORIGINAL MEDICAL REPORT EXCERPT:")
print(sample_report[:700] + "...\n")
print("-" * 80)
print("GROUND TRUTH (HUMAN WRITTEN SUMMARY):")
print(human_summary + "\n")
print("-" * 80)
print("AI GENERATED SUMMARY:")
print(generated_summary)
print("="*80 + "\n")

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,2.865770,28.335700,8.366100,17.983600,18.045400
2,No log,2.791492,29.507500,9.232700,18.438000,18.463000
3,No log,2.757843,29.277900,9.086500,18.450200,18.569800
4,No log,2.739233,29.981200,9.472900,18.945000,19.025900
5,No log,2.734737,29.926200,9.492400,18.968000,19.059300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



ORIGINAL MEDICAL REPORT EXCERPT:
the detection and quantification of tumour - specific rearrangements are important issues in cancer research and in clinical diagnosis of tumours . 
 in particular , its significance became obvious for haematological malignancies that exhibit characteristic translocations in specific tumour subgroups . 
 although gene rearrangements are typical for haematological malignancies , they also may occur in solid tumours as characteristic changes . 
 this has been shown for ret / ptc rearrangements in papillary thyroid carcinoma ( ptc ) that fuse the ret proto - oncogene to a variety of constitutively expressed partner genes ( for review see zitzelsberger ) . 
 this was further improved by the deve...

--------------------------------------------------------------------------------
GROUND TRUTH (HUMAN WRITTEN SUMMARY):
structural genomic rearrangements are frequent findings in human cancers . 
 therefore , papillary thyroid carcinomas ( ptcs ) were investigat